# Vector Search — Similarity Search & RAG

Tests two things:
1. **Similarity search** — retrieve relevant knowledge base documents via the vector index
2. **RAG** — pass retrieved context to the LLM to answer a question about inbound planning

Prerequisites: wheel installed on the cluster, `process_data_job` has run successfully.

In [0]:
from openai import OpenAI
from pyspark.sql import SparkSession

from inbound_planning.config import ProjectConfig
from inbound_planning.vector_search import VectorSearchManager

In [0]:
spark = SparkSession.builder.getOrCreate()

cfg = ProjectConfig.from_yaml("/Workspace/Users/aganbarov1994@gmail.com/.bundle/llmops-databricks-course-aliganbarov/dev/files/project_config.yml")
vs = VectorSearchManager(config=cfg)

print(f"Catalog  : {cfg.catalog}")
print(f"Schema   : {cfg.schema}")
print(f"VS index : {vs.index_name}")
print(f"LLM      : {cfg.llm_endpoint}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Catalog  : llmops_forecasting_dev
Schema   : inbound_planning
VS index : llmops_forecasting_dev.inbound_planning.knowledge_base_index
LLM      : databricks-meta-llama-3-3-70b-instruct


## 1. Similarity Search

Query the vector index directly and inspect the raw results.

In [0]:
query = "Which warehouses are at risk of exceeding capacity?"

results = vs.search(query=query, num_results=5)
print(f"Query: {query!r}\n")

data_array = results.get("result", {}).get("data_array", [])
columns = ["id", "text", "warehouse", "week", "doc_type"]

print(f"{'#':<3}  {'warehouse':<20}  {'week':<6}  {'doc_type':<18}  text preview")
print("-" * 90)
for i, row in enumerate(data_array, 1):
    doc = dict(zip(columns, row))
    preview = (doc["text"] or "").replace("\n", " ")
    print(f"{i:<3}  {str(doc['warehouse']):<20}  {str(doc['week']):<6}  {str(doc['doc_type']):<18}  {preview}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Query: 'Which warehouses are at risk of exceeding capacity?'

#    warehouse             week    doc_type            text preview
------------------------------------------------------------------------------------------
1    Augsburg              4.0     warehouse           Week 4 - Warehouse Augsburg  Forecasted inbound volume: 94,867 units. Capacity: 74,563 units. Utilization: 127%.  This warehouse is significantly over capacity.
2    Essen                 4.0     warehouse           Week 4 - Warehouse Essen  Forecasted inbound volume: 84,288 units. Capacity: 72,354 units. Utilization: 116%.  This warehouse is significantly over capacity.
3    Bielefeld             4.0     warehouse           Week 4 - Warehouse Bielefeld  Forecasted inbound volume: 89,859 units. Capacity

### Optional: filter by doc_type or warehouse

In [0]:
# Filter to network summaries only
network_results = vs.search(
    query="forecast trend across all warehouses",
    num_results=3,
    filters={"doc_type": "network_summary"},
)

print("Network summary results:")
for row in network_results.get("result", {}).get("data_array", []):
    doc = dict(zip(["id", "text", "warehouse", "week", "doc_type"], row))
    print(f"\n[Week {doc['week']}] {doc['text'][:200]}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Network summary results:

[Week 4.0] Week 4 - Network Summary

Warehouses over capacity:
Augsburg, Bielefeld, Braunschweig, Bremen, Dortmund, Düsseldorf, Essen, Kiel, Leipzig, Mannheim, Mönchengladbach, Wuppertal

Warehouses near capacit

[Week 11.0] Week 11 - Network Summary

Warehouses over capacity:
Aachen, Augsburg, Bielefeld, Bremen, Dortmund, Dresden, Duisburg, Essen, Hannover, Kiel, Leipzig, Mannheim, Mönchengladbach, Wiesbaden, Wuppertal



[Week 5.0] Week 5 - Network Summary

Warehouses over capacity:
Aachen, Augsburg, Bielefeld, Braunschweig, Bremen, Dortmund, Düsseldorf, Essen, Hannover, Kiel, Leipzig, Mannheim, Mönchengladbach, Münster, Wuppert


### Try different queries

In [0]:
test_queries = [
    "Berlin warehouse capacity utilization",
    "seasonal demand spike autumn",
    "low utilization warehouses",
]

for q in test_queries:
    hits = vs.search(query=q, num_results=3)
    rows = hits.get("result", {}).get("data_array", [])
    print(f"\nQuery: {q!r}")
    for row in rows:
        doc = dict(zip(["id", "text", "warehouse", "week", "doc_type"], row))
        print(f"  [{doc['doc_type']}] {doc['warehouse']} / week {doc['week']} — {(doc['text'] or '')[:100]}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Query: 'Berlin warehouse capacity utilization'
  [warehouse] Berlin / week 4.0 — Week 4 - Warehouse Berlin

Forecasted inbound volume: 62,220 units.
Capacity: 112,677 units.
Utiliza
  [warehouse] Berlin / week 5.0 — Week 5 - Warehouse Berlin

Forecasted inbound volume: 78,094 units.
Capacity: 112,677 units.
Utiliza
  [warehouse] Berlin / week 12.0 — Week 12 - Warehouse Berlin

Forecasted inbound volume: 100,290 units.
Capacity: 112,677 units.
Utili
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

Query: 'seasonal demand spike autumn'
  [warehouse] Augsburg / week 9.0 — Week 9 - Warehouse Augsburg

Forecasted

## 2. RAG — LLM on Top of Retrieved Context

Retrieve the most relevant documents and pass them to the LLM as context.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
llm_client = OpenAI(
    api_key=w.tokens.create(lifetime_seconds=1200).token_value,
    base_url=f"{w.config.host}/serving-endpoints",
)

In [0]:
def build_context(results: dict) -> str:
    """Format similarity search results into a context block for the prompt."""
    rows = results.get("result", {}).get("data_array", [])
    columns = ["id", "text", "warehouse", "week", "doc_type"]
    parts = []
    for i, row in enumerate(rows, 1):
        doc = dict(zip(columns, row))
        parts.append(
            f"[{i}] Type: {doc['doc_type']} | Warehouse: {doc['warehouse']} | Week: {doc['week']}\n"
            f"{doc['text']}"
        )
    return "\n\n---\n\n".join(parts)


def rag_query(question: str, num_docs: int = 5, filters: dict | None = None) -> str:
    """Answer a question using retrieved knowledge base documents.

    Args:
        question: Natural-language question about inbound planning.
        num_docs: Number of documents to retrieve.
        filters: Optional column filters passed to similarity_search.

    Returns:
        LLM answer string.
    """
    # Step 1: retrieve
    search_results = vs.search(query=question, num_results=num_docs, filters=filters)
    context = build_context(search_results)

    # Step 2: build prompt
    messages = [
        {"role": "system", "content": cfg.system_prompt},
        {
            "role": "user",
            "content": (
                f"Use the following inbound planning documents to answer the question.\n\n"
                f"CONTEXT:\n{context}\n\n"
                f"QUESTION: {question}\n\n"
                f"Answer concisely, referencing specific warehouses and weeks where relevant."
            ),
        },
    ]

    # Step 3: call LLM
    response = llm_client.chat.completions.create(
        model=cfg.llm_endpoint,
        messages=messages,
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content

### Ask a question

In [0]:
question = "Which warehouses are forecasted to exceed capacity in the coming weeks, and what is driving that?"

answer = rag_query(question, num_docs=6)
print(f"Q: {question}\n")
print(f"A: {answer}")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Q: Which warehouses are forecasted to exceed capacity in the coming weeks, and what is driving that?

A: Warehouses Wiesbaden (Week 9 and 11), Düsseldorf (Week 4), Bremen (Week 4), and Hannover (Week 10) are forecasted to exceed capacity. The drivers include strong increases in inbound volume (Wiesbaden in Weeks 6 and 9) and relatively stable high volumes (Hannover in Week 10 and Wiesbaden in Week 11, following a decrease).


### Try more questions

In [0]:
questions = [
    "Which warehouses have utilization below 70% and may have surplus capacity?",
    "Summarise the network-level forecast trend for the next 10 weeks.",
    "Are there any anomalies or unusual spikes in the forecast data?",
]

for q in questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {q}")
    print(f"A: {rag_query(q, num_docs=50)}")


Q: Which warehouses have utilization below 70% and may have surplus capacity?
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
A: None of the provided documents show a warehouse with utilization below 70%. However, some warehouses are operating close to capacity (utilization below 100%) in certain weeks, such as:

- Warehouse Wiesbaden in Week 4 (utilization: 90%) [9]
- Warehouse Aachen in Week 4 (utilization: 91%) [12]
- Warehouse Hannover in Week 4 (utilization: 96%) [20]

Q: Summarise the network-level forecast trend for the next 10 weeks.
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
A: Based on the provided data, the network-level forecast trend for